# AXI test 47: three arms, plus a summary arm, on the 50 frozen tau2-bench retail tasks (Gemini 3.8 Flash)
Pre-registered page: `docs/TAU2_PILOT.md` (the plan, the gate, the predictions, all written before this run). Same model on both sides of the comparison: the benchmark's own agent (`B`), the same agent with the host's gate (`G`), and the host-held state with the same gate (`A`). An extra arm `S` is the benchmark's agent with the chat rolled up by the model (a summary baseline). The benchmark's own user simulator and evaluator score every run.

**What to know**
* **No GPU.** The free CPU runtime is enough. Runtime > Run all, top to bottom.
* **A key.** Put your key into Colab **Secrets** under the name `GEMINI_API_KEY` (or `GOOGLE_API_KEY`) and switch **Notebook access** on (it is per notebook). The key is held in memory and is never printed or written to a file.
* **Cost and time.** Smoke test (task 0, four arms): a few cents; it also gives the real cost and time per task, which replace the estimate here. Full run (the benchmark's own `retail_50`, 50 tasks, four arms, 200 runs): an estimate of about $15 to $40 at list prices, worst case about $65; Google's prices double on 1 Jan 2027. **There is no spending cap in the code: set a budget alert or limit in Google billing before Cell 5.** **It needs a key with billing on** (a free-tier key allows 20 requests a day for this model, which the smoke test alone uses up; the runner then stops and says so). Set `RPM = 0` in Cell 2. The full run takes from one to several hours; if it stops (quota, a closed tab), run Cell 5 again and it continues from the saved files (`--auto-resume`).
* The pass counts are a first look (50 tasks, one trial each); the page's sign-test rule says when a difference counts. The token counts are the measurement. The user simulator is Gemini too, so the numbers are not comparable with the benchmark's leaderboard.

### Cell 1: get this repository and the benchmark at the pinned commit (about 2 to 4 minutes)

In [ ]:
%cd /content
!rm -rf /content/axi /content/tau2-bench
!git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
!git clone -q https://github.com/sierra-research/tau2-bench /content/tau2-bench
!cd /content/tau2-bench && git checkout -q 768f3d8fe8528b4f32a5096f9ed008ae50f48ead && git log -1 --format='%H %cd'
!pip -q install -e /content/tau2-bench
import os
assert os.path.exists("/content/axi/python/axi/experiments/tau2_pilot_run.py"), "test 47 is not on GitHub yet"
print("ready")

### Cell 2: the key (Colab Secrets, name `GEMINI_API_KEY` or `GOOGLE_API_KEY`)

In [ ]:
import os
from google.colab import userdata
key = None
for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):  # the saved secret may carry either name; the key itself is never printed
    try:
        key = userdata.get(name)
        break
    except Exception:  # no such secret, or Notebook access is off for it
        continue
os.environ["GEMINI_API_KEY"] = key or ""
print("key set:", bool(key))
RPM = 0  # 0 = no waiting (a key with billing on). A key with only a per-minute cap: set it to 4 or 5 below that cap.
CONCURRENCY = 1 if RPM else 2

### Cell 3: the offline checks (no model call): the gate, the held state, the glue and the summary

In [ ]:
%cd /content/axi
!PYTHONPATH=python python3 python/tests/test_tau2_retail_core.py | tail -2
!PYTHONPATH=python python3 python/tests/test_tau2_retail_agents.py | tail -2
!PYTHONPATH=python python3 python/tests/test_tau2_pilot_run.py | tail -2

### Cell 4: smoke test, task 0 on all four arms (a few cents)
If this fails, stop and send the error text; do not run Cell 5.

In [ ]:
%cd /content/axi/python
!python3 -m axi.experiments.tau2_pilot_run run --arms B G A S --tasks 0 --tag smoke --concurrency {CONCURRENCY} --rpm {RPM}

In [ ]:
!python3 -m axi.experiments.tau2_pilot_run summary --tag smoke --tasks 0 2>&1 | tail -40

### Cell 5: the full run, the 50 tasks of `retail_50` on four arms (one to several hours; run it again to continue if it stops)

In [ ]:
!python3 -m axi.experiments.tau2_pilot_run run --arms B G A S --tag t47 --concurrency {CONCURRENCY} --rpm {RPM}

### Cell 6: the tables, the paired comparisons and the eight readings of the plan page

In [ ]:
!python3 -m axi.experiments.tau2_pilot_run summary --tag t47 2>&1 | tail -60
import json
print(json.dumps(json.load(open("t47_summary.json"))))

In [ ]:
from google.colab import files
files.download("t47_summary.json")